In [ ]:
%matplotlib widget
import pandas 
import pathlib
import ipywidgets
from ipyfilechooser import FileChooser

import numpy

import read_h5

import magic_graphs
import scipp
import scippnexus
import plopp
import widget_code_cell
from ui import spinner

import operations_with_da

In [ ]:
# Global
DF_FILES = pandas.DataFrame(columns=["filename", "size_gb", "modified", "full_path"])

DG_DATA = scipp.DataGroup()

DG_CALC = scipp.DataGroup()

MAGIC_GRAPH = {**magic_graphs.graph_detector, **magic_graphs.graph_qvec, **magic_graphs.graph_hkl}

D_DISPLAY_HIST_UNIT = {
    "event_gamma": ("deg", "rad"), 
    "event_nu": ("deg", "rad"), 
    "toa": ("ms", "s"), 
    "wavelength": ("Å", "Å"), 
    "norm_Q": ("1/Å", "1/Å"), 
    "two_theta": ("deg", "rad"), 
    "event_two_theta": ("deg", "rad"), 
    "event_phi": ("deg", "rad"), 
    "sin_theta": ("", ""), 
    "d_space": ("Å", "Å"),
    "Qx": ("1/Å", "1/Å"), 
    "Qy": ("1/Å", "1/Å"), 
    "Qz": ("1/Å", "1/Å"), 
    "h": ("", ""), 
    "k": ("", ""), 
    "l": ("", ""), 
    "h_reduced": ("", ""), 
    "k_reduced": ("", ""), 
    "l_reduced": ("", ""),
    "event_position_global": ("mm", "m"), 
    "Q_vec_rot": ("1/Å","1/Å"), 
    "hkl_vec": ("", ""),
    }

# It should be given in DISPLAY UNITS
D_DEFAULT_STEP = {
    'toa': scipp.scalar(0.050, unit='ms'), 
    'event_gamma': scipp.scalar(0.15, unit='deg'), 
    'event_nu': scipp.scalar(0.33, unit='deg'), 
    'two_theta': scipp.scalar(0.20, unit='deg'), 
    'event_two_theta': scipp.scalar(0.20, unit='deg'), 
    'event_phi': scipp.scalar(0.50, unit='deg'), 
    'sin_theta': scipp.scalar(0.001, unit=''), 
    'd_space': scipp.scalar(0.005, unit='Å'), 
    'wavelength': scipp.scalar(0.005, unit='Å'), 
    'norm_Q': scipp.scalar(0.005, unit='1/Å'),
    'Qx': scipp.scalar(0.005, unit='1/Å'),
    'Qy': scipp.scalar(0.005, unit='1/Å'),
    'Qz': scipp.scalar(0.005, unit='1/Å'),
    }


D_DETECTOR_KEYS = {'da': 'detector_a', 'db': 'detector_b'}

SPATIAL_AXIS_NAMES = ["3D Laue", "Q Space", "HKL Space"]

S_SPATIAL_AXIS_NAMES = ["event_position_global", "Q_vec_rot", "hkl_vec"]

AXIS_NAMES = ["None", "γ", "ν", "TOF", "λ", "Q", "2θ", "2θ_l", "φ_l", "sin θ", "d", "Q_x", "Q_y", "Q_z", "h", "k", "l", "h_reduced", "k_reduced", "l_reduced", ]

S_AXIS_NAMES = ["", "event_gamma", "event_nu", "toa", "wavelength", "norm_Q", "two_theta", "event_two_theta", "event_phi", "sin_theta", "d_space", "Qx", "Qy", "Qz", "h", "k", "l", "h_reduced", "k_reduced", "l_reduced", ] # Names like in MAGiC graph




In [ ]:
# Functions
def file_metadata(path: pathlib.Path):
    return {
        "filename": path.name,
        "size_gb": round(path.stat().st_size / 1e9, 2),
        "modified": pandas.to_datetime(path.stat().st_mtime, unit="s"),
        "full_path": str(path.resolve()),
    }


def display_center(*graphs):
    return display(
        ipywidgets.VBox(graphs, layout=ipywidgets.Layout(align_items="center"))
    )


def range_block(axis_label):
    """Create one range block with min/max/step."""
    
    min_val = ipywidgets.FloatText(
        description="Min:",
        value=None,
        layout=ipywidgets.Layout(width="200px")
    )
    
    max_val = ipywidgets.FloatText(
        description="Max:",
        value=None,
        layout=ipywidgets.Layout(width="200px")
    )
    
    step_val = ipywidgets.FloatText(
        description="Step:",
        value=None,
        layout=ipywidgets.Layout(width="200px")
    )
    
    box = ipywidgets.VBox([
        ipywidgets.HTML(f"<b>{axis_label}</b>"),
        ipywidgets.HBox([min_val, max_val, step_val])
    ])
    
    return {
        "widget": box,
        "min": min_val,
        "max": max_val,
        "step": step_val
    }


def int_block(axis_label):
    """Create one int block with integer value."""
    
    int_val = ipywidgets.FloatText(
        description="",
        value=0,
        layout=ipywidgets.Layout(width="200px")
    )
    
    box = ipywidgets.VBox([
        ipywidgets.HTML(f"<b>{axis_label}</b>"),
        int_val, 
    ])
    
    return {
        "widget": box,
        "int": int_val,
    }


def get_sc_scalar_from_block(d_block:dict, s_key:str, s_name:str=""):
    if s_name == "":
        s_name = s_key
    widget_val = d_block[s_key].value
    
    if s_name in D_DISPLAY_HIST_UNIT.keys():
        (s_unit_display, s_unit_hist) = D_DISPLAY_HIST_UNIT[s_name]
        res = scipp.scalar(widget_val, unit=s_unit_display).to(unit=s_unit_hist)
    else:
        print(f'Parameters {s_name:} is not found in D_DISPLAY_HIST_UNIT')
        res = scipp.scalar(widget_val, unit=None)
    return res

def default_step_block(label:str):
    d_out = {}
    l_widget = [ipywidgets.HTML(f"<b>{label:}</b>"), ] 
    for s_key, sc_value in D_DEFAULT_STEP.items():
        if s_key in S_AXIS_NAMES:
            s_name = AXIS_NAMES[S_AXIS_NAMES.index(s_key)]
        else:
            s_name = s_key
        step_widget = ipywidgets.FloatText(
        description=f"Δ{s_name:} ({sc_value.unit:})",
        value=sc_value.value,
        layout=ipywidgets.Layout(width="200px")
        )
        d_out[s_key] = step_widget
        l_widget.append(step_widget)
    box = ipywidgets.VBox(l_widget)
    d_out['widget'] = box
    return d_out

    
def spatial_axis_block(axis_label):
    """Create one axis block with dropdown + min/max/step."""
    
    dropdown = ipywidgets.Dropdown(
        options=SPATIAL_AXIS_NAMES,
        description="",
        layout=ipywidgets.Layout(width="250px")
    )
    
    min_x_val = ipywidgets.FloatText(
        description="Min X:",
        value=None,
        layout=ipywidgets.Layout(width="200px")
    )
    
    max_x_val = ipywidgets.FloatText(
        description="Max X:",
        value=None,
        layout=ipywidgets.Layout(width="200px")
    )
    
    step_x_val = ipywidgets.FloatText(
        description="Step X:",
        value=None,
        layout=ipywidgets.Layout(width="200px")
    )

    min_y_val = ipywidgets.FloatText(
        description="Min Y:",
        value=None,
        layout=ipywidgets.Layout(width="200px")
    )
    
    max_y_val = ipywidgets.FloatText(
        description="Max Y:",
        value=None,
        layout=ipywidgets.Layout(width="200px")
    )
    
    step_y_val = ipywidgets.FloatText(
        description="Step Y:",
        value=None,
        layout=ipywidgets.Layout(width="200px")
    )

    min_z_val = ipywidgets.FloatText(
        description="Min Z:",
        value=None,
        layout=ipywidgets.Layout(width="200px")
    )
    
    max_z_val = ipywidgets.FloatText(
        description="Max Z:",
        value=None,
        layout=ipywidgets.Layout(width="200px")
    )
    
    step_z_val = ipywidgets.FloatText(
        description="Step Z:",
        value=None,
        layout=ipywidgets.Layout(width="200px")
    )

    box = ipywidgets.VBox([
        ipywidgets.HTML(f"<b>{axis_label}</b>"),
        dropdown,
        ipywidgets.HBox([min_x_val, max_x_val, step_x_val]),
        ipywidgets.HBox([min_y_val, max_y_val, step_y_val]),
        ipywidgets.HBox([min_z_val, max_z_val, step_z_val]),
    ])
    
    return {
        "widget": box,
        "dropdown": dropdown,
        "min_x": min_x_val, "max_x": max_x_val, "step_x": step_x_val,
        "min_y": min_y_val, "max_y": max_y_val, "step_y": step_y_val,
        "min_z": min_z_val, "max_z": max_z_val, "step_z": step_z_val,
    }


def axis_block(axis_label):
    """Create one axis block with dropdown + min/max/step."""
    
    dropdown = ipywidgets.Dropdown(
        options=AXIS_NAMES,
        description="",
        layout=ipywidgets.Layout(width="250px")
    )
    
    min_val = ipywidgets.FloatText(
        description="Min:",
        value=None,
        layout=ipywidgets.Layout(width="200px")
    )
    
    max_val = ipywidgets.FloatText(
        description="Max:",
        value=None,
        layout=ipywidgets.Layout(width="200px")
    )
    
    step_val = ipywidgets.FloatText(
        description="Step:",
        value=None,
        layout=ipywidgets.Layout(width="200px")
    )
    
    box = ipywidgets.VBox([
        ipywidgets.HTML(f"<b>{axis_label}</b>"),
        dropdown,
        ipywidgets.HBox([min_val, max_val, step_val])
    ])
    
    return {
        "widget": box,
        "dropdown": dropdown,
        "min": min_val,
        "max": max_val,
        "step": step_val
    }


def move_data_from_dg_to_da(dg_magic, data_event):
    operations_with_da.assign_dg_to_da_coords(dg_magic['sample'], data_event, prefix="sample")
    data_event.coords['tp_position'] = dg_magic['tp_position']
    data_event.coords['source_position'] = dg_magic['source_position']
    data_event.coords['delta_L'] = dg_magic['delta_L']
    data_event.coords['delta_t'] = dg_magic['delta_t']
    return


def load_dg_by_df_row(df_row):
    file_path = pathlib.Path(df_row['full_path'])
    if file_path.suffix == ".h5":
        # McStas file
        d_out = read_h5.read_magic_from_nexus(file_path)
    else: # "hdf, nxs":
        d_out = scippnexus.load(file_path)
    s_key = str(df_row.name)
    DG_DATA[s_key] = d_out
    return
    

def take_dg_by_df_row(df_row):
    s_key = str(df_row.name)
    if not(s_key in DG_DATA.keys()):
        print(f"Loading file '{df_row['filename']}'.")
        load_dg_by_df_row(df_row)
    d_out = DG_DATA[s_key]
    return d_out


def take_name_from_dg_by_df_row(df_row, name:str):
    d_out = take_dg_by_df_row(df_row)
    res = d_out.get(name, None)
    return res


def upload_detector_reduced_calc_by_df_row(df_row, s_detector:str, da):
    dg_calc = take_dg_calc_by_df_row(df_row)
    s_key = f'{s_detector:}_reduced'
    dg_calc[s_key] = da


def take_dg_calc_by_df_row(df_row):
    s_key = str(df_row.name)
    if not(s_key in DG_CALC.keys()):
        DG_CALC[s_key] = scipp.DataGroup()
    return DG_CALC[s_key]


def take_detector_a_laue_hist_by_df_row(df_row):
    dg_calc = take_dg_calc_by_df_row(df_row)
    s_key = 'detector_a_laue_hist'
    if not(s_key in dg_calc.keys()):
        print(f"Calculating '{s_key:}'.")
        detector_a = take_name_from_dg_by_df_row(df_row, 'detector_a')
        if detector_a is None:
            return None
        detector_a_laue_hist = operations_with_da.da_to_laue_hist(detector_a, factor_border=0.00)
        dg_calc[s_key] = detector_a_laue_hist
    return dg_calc[s_key]


def take_detector_b_laue_hist_by_df_row(df_row):
    dg_calc = take_dg_calc_by_df_row(df_row)
    s_key = 'detector_b_laue_hist'
    if not(s_key in dg_calc.keys()):
        print(f"Calculating '{s_key:}'.")
        detector_b = take_name_from_dg_by_df_row(df_row, 'detector_b')
        if detector_b is None:
            return None
        detector_b_laue_hist = operations_with_da.da_to_laue_hist(detector_b, factor_border=0.00)
        dg_calc[s_key] = detector_b_laue_hist
    return dg_calc[s_key]


def take_detector_a_image_by_df_row(df_row):
    dg_calc = take_dg_calc_by_df_row(df_row)
    s_key = 'detector_a_image'
    if not(s_key in dg_calc.keys()):
        print(f"Calculating '{s_key:}'.")
        detector_a = take_name_from_dg_by_df_row(df_row, 'detector_a')
        detector_a_image = operations_with_da.da_to_2d_hist(detector_a, factor_border=0.00)
        dg_calc[s_key] = detector_a_image
    return dg_calc[s_key]


def take_detector_b_image_by_df_row(df_row):
    dg_calc = take_dg_calc_by_df_row(df_row)
    s_key = 'detector_b_image'
    if not(s_key in dg_calc.keys()):
        print(f"Calculating '{s_key:}'.")
        detector_b = take_name_from_dg_by_df_row(df_row, 'detector_b')
        detector_b_image = operations_with_da.da_to_2d_hist(detector_b, factor_border=0.00)
        dg_calc[s_key] = detector_b_image
    return dg_calc[s_key]



def get_flag_range_and_points_number(range, default_min, default_max):
    """range is (min, max, step)
    """
    points_number = 0
    flag_range = range[2] > scipp.scalar(0, unit=default_min.unit)
    val_min = default_min
    val_max = default_max
    if range[0]!=scipp.scalar(0, unit=default_min.unit):
        val_min = range[0]
    if range[1]!=scipp.scalar(0, unit=default_min.unit):
        val_max = range[1]
    if flag_range:
        points_number = int(numpy.round(((val_max-val_min)/range[2]).value, 0)) + 1
    return flag_range, (val_min, val_max, points_number)


def assign_peaks_to_reduced_data(df_row, da_peaks:scipp.DataArray, s_detector:str):
    da_peaks_gamma_nu_toa = da_peaks
    if not(da_peaks_gamma_nu_toa is None):
        detector = take_detector_reduced_calc_by_df_row(df_row, s_detector, flag_calc=False)
        if not detector is None:
            detector = detector.transform_coords(
                ("event_gamma", "event_nu", "event_position_global"),graph=magic_graphs.graph_detector, rename_dims=False,)
            range_sigma = 5
            operations_with_da.assign_event_peak_to_da(
                detector, 
                da_peaks_gamma_nu_toa.coords["toa"], 
                da_peaks_gamma_nu_toa.coords["event_gamma"], 
                da_peaks_gamma_nu_toa.coords["event_nu"], 
                da_peaks_gamma_nu_toa.coords["toa_sigma"], 
                da_peaks_gamma_nu_toa.coords["event_gamma_sigma"], 
                da_peaks_gamma_nu_toa.coords["event_nu_sigma"], 
                range_sigma)
            upload_detector_reduced_calc_by_df_row(df_row, s_detector, detector)



In [ ]:
# Widgets

fc_path_input = FileChooser(
    path = pathlib.Path('.').resolve(),
    # show_only_dirs=True,
    title='',
    layout=ipywidgets.Layout(width="95%"),
    )


load_button = ipywidgets.Button(
    description="Load",
    button_style="info",
    layout=ipywidgets.Layout(width="120px")
)


clean_button = ipywidgets.Button(
    description="Clean All",
    button_style="danger",
    layout=ipywidgets.Layout(width="120px")
)


clean_data_button = ipywidgets.Button(
    description="Data",
    button_style="danger",
    layout=ipywidgets.Layout(width="120px")
)


clean_calc_button = ipywidgets.Button(
    description="Calculations",
    button_style="danger",
    layout=ipywidgets.Layout(width="120px")
)


extra_reduction_checkbox = ipywidgets.Checkbox(
    value=False,
    description="More",
    disabled=False,
    indent=False
)

time_reduction_block = range_block("TOF Reduction (ms)")
time_reduction_block['step'].value=0.01
# time_reduction_block['max'].value=numpy.round(2*1000/14, 2)
time_reduction_block['widget'].layout.display= "none"

count_reduction_block = int_block("Count Reduction")
count_reduction_block['widget'].layout.display= "none"

step_reduction_block = default_step_block('Default Step')
step_reduction_block['widget'].layout.display= "none"

reduce_button = ipywidgets.Button(
    description="Reduce",
    button_style="success",
    layout=ipywidgets.Layout(width="120px")
)
reduce_button.layout.display= "none"


tof_adjust_button = ipywidgets.Button(
    description="Adjust Range",
    button_style="info",
    layout=ipywidgets.Layout(width="120px")
)
tof_adjust_button.layout.display= "none"

add_output = ipywidgets.Output()


file_selector = ipywidgets.SelectMultiple(
    options=DF_FILES["filename"].tolist(),
    description="",
    layout=ipywidgets.Layout(width="80%", height="200px")
)


filter = ipywidgets.Text(
    value="",
    placeholder="Filter",
    description="",
    layout=ipywidgets.Layout(width="80%")
)


plot_laue_button = ipywidgets.Button(
    description="3D Laue",
    button_style="info",
    layout=ipywidgets.Layout(width="150px")
)


plot_hist_3d_button = ipywidgets.Button(
    description="γ-ν-TOF",
    button_style="info",
    layout=ipywidgets.Layout(width="150px")
)


plot_hist_2d_button = ipywidgets.Button(
    description="TOF-2θ",
    button_style="info",
    layout=ipywidgets.Layout(width="150px")
)

plot_hist_1d_button = ipywidgets.Button(
    description="TOF",
    button_style="info",
    layout=ipywidgets.Layout(width="150px")
)


extra_plots_checkbox = ipywidgets.Checkbox(
    value=False,
    description="More",
    disabled=False,
    indent=False
)


detector_type_radio_button = ipywidgets.RadioButtons(
    options=['Detector A', 'Detector B'],
    layout={'width': '158px', 'display': 'none'},
    description='',
    disabled=False
)

normalization_radio_button = ipywidgets.RadioButtons(
    options=['No Normalization', 'Per Monitor', 'Per Vanadium'],
    layout={'width': '150px', 'display': 'none'},
    description='',
    disabled=False
)


display_peaks_checkbox = ipywidgets.Checkbox(
    value=False,
    layout={'display': 'none'},
    description="Display Peaks",
    disabled=False,
    indent=False
)


plot_graph_button = ipywidgets.Button(
    description="Display Graph",
    button_style="info",
    layout={'width': '150px', 'display': 'none'},
)


plot_monitor_button = ipywidgets.Button(
    description="Cave Monitor",
    button_style="info",
    layout={'width': '150px', 'display': 'none'},
)

# Predefined axis names

# Build all three axes
axis_x_block = axis_block("Axis X")
axis_y_block = axis_block("Axis Y")
axis_z_block = axis_block("Axis Z")
axis_spatial_block = spatial_axis_block("Event space")

axis_x_block['widget'].layout.display= "none"
axis_y_block['widget'].layout.display= "none"
axis_z_block['widget'].layout.display= "none"
axis_spatial_block['widget'].layout.display= "none"

find_peaks_button = ipywidgets.Button(
    description="Find Peaks",
    button_style="success",
    layout=ipywidgets.Layout(width="150px")
)

extra_find_peaks_checkbox = ipywidgets.Checkbox(
    value=False,
    description="More",
    disabled=False,
    indent=False
)


threshold_boundfloattext = ipywidgets.BoundedFloatText(
    value=0.1,
    min=0.000,
    max=1.000,
    step=0.05,
    placeholder="From 0. to 1.",
    description="Threshold",
    layout={'width':"150px", "display":"none"}
)

binary_dilation_boundinttext= ipywidgets.BoundedIntText(
    value=3,
    min=1,
    max=10000,
    step=1,
    description="Dilation",
    layout={'width':"150px", "display":"none"}
)


fc_load_peaks = FileChooser(
    path = pathlib.Path('.').resolve(),
    # show_only_dirs=True,
    title='',
    layout={'width':"460px", 'display': 'none'},
    )

load_peaks_button= ipywidgets.Button(
    description="Load Peaks",
    button_style="info",
    layout={'width':"150px", 'display': 'none'},
    )


load_peaks_detector_type_radio_button = ipywidgets.RadioButtons(
    options=['Detector A', 'Detector B'],
    layout={'display': 'none'},
    description='',
    orientation='horizontal',
    disabled=False
)

save_calculations_button = ipywidgets.Button(
    description="Save Calc.",
    button_style="success",
    layout=ipywidgets.Layout(width="150px")
)

load_output = ipywidgets.Output()

In [ ]:
# Functions that requires widgets
def get_s_detector_and_normalization():
    s_detector = "".join([hh[0] for hh in detector_type_radio_button.value.strip().split()]).lower()
    s_normalization = "".join([hh[0] for hh in normalization_radio_button.value.strip().split()]).lower()
    return (s_detector, s_normalization)


def get_s_1d_ax():
    s_ax_x = S_AXIS_NAMES[AXIS_NAMES.index(axis_x_block['dropdown'].value)]
    if s_ax_x == "":
        s_ax_x = "toa"
    return s_ax_x


def get_s_2d_ax():
    s_ax_y = S_AXIS_NAMES[AXIS_NAMES.index(axis_y_block['dropdown'].value)]
    if s_ax_y == "":
        s_ax_y = "two_theta"
    return get_s_1d_ax(), s_ax_y


def get_s_3d_ax():
    s_ax_x = S_AXIS_NAMES[AXIS_NAMES.index(axis_x_block['dropdown'].value)]
    if s_ax_x == "":
        s_ax_x = "event_gamma"
    s_ax_y = S_AXIS_NAMES[AXIS_NAMES.index(axis_y_block['dropdown'].value)]
    if s_ax_y == "":
        s_ax_y = "event_nu"
    s_ax_z = S_AXIS_NAMES[AXIS_NAMES.index(axis_z_block['dropdown'].value)]
    if s_ax_z == "":
        s_ax_z = "toa"
    return s_ax_x, s_ax_y, s_ax_z


def get_range_time_reduction():
    min_val = get_sc_scalar_from_block(time_reduction_block, 'min', 'toa')
    max_val = get_sc_scalar_from_block(time_reduction_block, 'max', 'toa')
    step_val = get_sc_scalar_from_block(time_reduction_block, 'step', 'toa')
    return (min_val, max_val, step_val)


def get_range_1d_ax():
    s_ax =get_s_1d_ax()
    min_val = get_sc_scalar_from_block(axis_x_block, 'min', s_ax)
    max_val = get_sc_scalar_from_block(axis_x_block, 'max', s_ax)
    step_val = get_sc_scalar_from_block(axis_x_block, 'step', s_ax)
    if s_ax in step_reduction_block.keys():
        step_min = get_sc_scalar_from_block(step_reduction_block, s_ax, s_ax)
        if step_val < step_min:
            step_val = step_min
    return (min_val, max_val, step_val)


def get_range_2d_ax():
    range_x = get_range_1d_ax()
    s_ax =get_s_2d_ax()[1]
    min_val = get_sc_scalar_from_block(axis_y_block, 'min', s_ax)
    max_val = get_sc_scalar_from_block(axis_y_block, 'max', s_ax)
    step_val = get_sc_scalar_from_block(axis_y_block, 'step', s_ax)
    if s_ax in step_reduction_block.keys():
        step_min = get_sc_scalar_from_block(step_reduction_block, s_ax, s_ax)
        if step_val < step_min:
            step_val = step_min
    return range_x, (min_val, max_val, step_val)


def get_range_3d_ax():
    range_x, range_y = get_range_2d_ax()
    s_ax_x, s_ax_y, s_ax_z =get_s_3d_ax()
    min_val_x = get_sc_scalar_from_block(axis_x_block, 'min', s_ax_x)
    max_val_x = get_sc_scalar_from_block(axis_x_block, 'max', s_ax_x)
    step_val_x = get_sc_scalar_from_block(axis_x_block, 'step', s_ax_x)
    if s_ax_x in step_reduction_block.keys():
        step_min_x = get_sc_scalar_from_block(step_reduction_block, s_ax_x, s_ax_x)
        if step_val_x < step_min_x:
            step_val_x = step_min_x

    min_val_y = get_sc_scalar_from_block(axis_y_block, 'min', s_ax_y)
    max_val_y = get_sc_scalar_from_block(axis_y_block, 'max', s_ax_y)
    step_val_y = get_sc_scalar_from_block(axis_y_block, 'step', s_ax_y)
    if s_ax_y in step_reduction_block.keys():
        step_min_y = get_sc_scalar_from_block(step_reduction_block, s_ax_y, s_ax_y)
        if step_val_y < step_min_y:
            step_val_y = step_min_y

    min_val_z = get_sc_scalar_from_block(axis_z_block, 'min', s_ax_z)
    max_val_z = get_sc_scalar_from_block(axis_z_block, 'max', s_ax_z)
    step_val_z = get_sc_scalar_from_block(axis_z_block, 'step', s_ax_z)
    if s_ax_z in step_reduction_block.keys():
        step_min_z = get_sc_scalar_from_block(step_reduction_block, s_ax_z, s_ax_z)
        if step_val_z < step_min_z:
            step_val_z = step_min_z

    return (min_val_x, max_val_x, step_val_x), (min_val_y, max_val_y, step_val_y), (min_val_z, max_val_z, step_val_z)


def take_detector_reduced_calc_by_df_row(df_row, s_detector:str, flag_calc:bool=False):
    dg_calc = take_dg_calc_by_df_row(df_row)
    s_key = f'{s_detector:}_reduced'
    if not(s_key in dg_calc.keys()) or flag_calc:
        print(f"Calculating '{s_key:}'.")
        s_detector_key = D_DETECTOR_KEYS[s_detector]
        detector = take_name_from_dg_by_df_row(df_row, s_detector_key)
        if detector is None:
            return None
        range_time = get_range_time_reduction()
        count_min = count_reduction_block['int'].value
        if range_time[2]>scipp.scalar(0,unit='s') and range_time[2]<scipp.scalar(1e-5,unit='s'): # 10 microsec it is dead time for detector A and detector B
             range_time = (range_time[0], range_time[1], scipp.scalar(1e-5,unit='s'))
        def_min = detector.bins.coords['toa'].min()
        def_max = detector.bins.coords['toa'].max()
        flag_range, mms = get_flag_range_and_points_number(range_time, def_min, def_max)
        if flag_range:
            s_unit = def_min.unit
            bin_toa = scipp.linspace(dim='toa', start =mms[0].to(unit=s_unit),stop=mms[1].to(unit=s_unit),num=mms[2], unit=s_unit, endpoint=True)
        else:
            bin_toa = 300
        detector_hist = detector.hist(toa=bin_toa)
        detector_hist.coords['toa'] = scipp.midpoints(detector_hist.coords['toa'])
        detector_event = scipp.flatten(detector_hist, to='event')
        dg_calc[s_key] = detector_event[(detector_event>count_min).data]
    return dg_calc[s_key]


def get_hist_nd(df_row, s_detector, s_normalization, dims:tuple[str, ...] = (), ranges:tuple = ()):
    """t_range is (min, max, step)
    """
    dg_calc = take_dg_calc_by_df_row(df_row)
    s_axes = "/".join(dims)
    s_key = f"hist_nd/{s_detector:}/{s_normalization:}/{s_axes:}"

    # if not(s_key in dg_calc.keys()):
    detector = take_detector_reduced_calc_by_df_row(df_row, s_detector=s_detector)
    if detector is None:
        return
    l_coord_keys = list(detector.coords.keys())
    if all([s_ax in l_coord_keys for s_ax in dims]):
        hh = detector
    else:
        dg_magic = take_dg_by_df_row(df_row)
        move_data_from_dg_to_da(dg_magic, detector)
        hh = detector.transform_coords(dims, graph=MAGIC_GRAPH)
    
    print(f"Calculating '{s_key:}'.")    
    d_bin = {s_ax:100 for s_ax in reversed(dims)}
    
    for s_ax, range in zip(dims, ranges):
        def_min = hh.coords[s_ax].min()
        def_max = hh.coords[s_ax].max()            
        flag_range, mms = get_flag_range_and_points_number(range, def_min, def_max)
        if flag_range:
            s_unit = hh.coords[s_ax].unit
            d_bin[s_ax] = scipp.linspace(dim=s_ax, start =mms[0].to(unit=s_unit).value,stop=mms[1].to(unit=s_unit).value,num=mms[2], unit=s_unit, endpoint=True)
    hist_nd = hh.hist(**d_bin)
    dg_calc[s_key] = hist_nd
    return dg_calc[s_key]


def get_peaks(df_row, s_detector:str, s_normalization:str, dims:tuple[str, ...] = (), ranges:tuple = (), threshold:float=0.1, binary_dilation:int=3):
    # Calculate histrogramm if it is not done yet
    # Calculate peaks based on histogram
    dg_calc = take_dg_calc_by_df_row(df_row)
    s_axes = "/".join(dims)
    s_key = f"hist_nd/{s_detector:}/{s_normalization:}/{s_axes:}"
    if not(s_key in dg_calc.keys()):
        da_hist = get_hist_nd(df_row, s_detector, s_normalization, dims=dims, ranges=ranges)
    else:
        da_hist = dg_calc[s_key]
    if da_hist is None:
        return None
    s_key = f"peaks/{s_detector:}/{s_normalization:}/{s_axes:}"
    # if not(s_key in dg_calc.keys()):
    print(f"Calculating '{s_key:}'.")    
    da_peaks = operations_with_da.find_peaks_hist(da_hist, threshold=threshold, binary_dilation=binary_dilation)
    dg_calc[s_key] = da_peaks
    return dg_calc[s_key]


def take_detector_peaks_gamma_nu_toa(df_row, s_detector:str):
    dg_calc = take_dg_calc_by_df_row(df_row)
    s_key = f'peaks/{s_detector:}/nn/event_gamma/event_nu/toa'
    if not(s_key in dg_calc.keys()) or True:
        print(f"Calculating '{s_key:}'.")
        
        step_gamma = get_sc_scalar_from_block(step_reduction_block, 'event_gamma')
        step_nu = get_sc_scalar_from_block(step_reduction_block, 'event_nu')
        step_toa = get_sc_scalar_from_block(step_reduction_block, 'toa')

        
        range_gamma = (scipp.scalar(0, unit=step_gamma.unit), scipp.scalar(0, unit=step_gamma.unit), step_gamma)
        range_nu = (scipp.scalar(0, unit=step_nu.unit), scipp.scalar(0, unit=step_nu.unit), step_nu)
        range_toa = (scipp.scalar(0, unit=step_toa.unit), scipp.scalar(0, unit=step_toa.unit), step_toa)
        
        threshold = threshold_boundfloattext.value
        binary_dilation = binary_dilation_boundinttext.value
        da_peaks = get_peaks(df_row, s_detector,'nn', dims=('event_gamma', 'event_nu', 'toa'), ranges=(range_gamma, range_nu, range_toa), threshold=threshold, binary_dilation=binary_dilation)
        if da_peaks is None:
            return None
    return dg_calc[s_key]



In [ ]:
# Callback
def refresh_selector():
    query = (filter.value or "").strip().lower()
    if DF_FILES.empty:
        file_selector.options = []
        return

    visible_indices = [
        idx for idx, name in DF_FILES["filename"].astype(str).items()
        if not query or query in name.lower()
    ]

    file_selector.options = [
        (DF_FILES.loc[idx, "filename"], idx) for idx in visible_indices
    ]


def on_filter_change(change):
    refresh_selector()


def on_add_clicked(b):
    add_output.clear_output()
    with add_output:
        p = pathlib.Path(fc_path_input.value).expanduser()

        if not p.exists():
            print("Path does not exist:", p)
            return
        fc_load_peaks.default_path = fc_path_input.selected_path
        global DF_FILES

        def already_in_df(path):
            return str(path.resolve()) in DF_FILES["full_path"].values

        # Directory → add all files
        if p.is_dir():
            files = [f for f in p.glob("*") if f.is_file() and f.suffix in ['.h5', '.nxs', '.hdf']]
            new_rows = []

            for f in files:
                if already_in_df(f):
                    print(f"Skipping (already in table): {f.name}")
                else:
                    new_rows.append(file_metadata(f))

            if new_rows:
                # Add new rows using .loc with the next available index
                start_idx = len(DF_FILES)
                for i, row in enumerate(new_rows):
                    DF_FILES.loc[start_idx + i] = row
                print(f"Added {len(new_rows)} new files from directory:", p)
            else:
                print("No new files to add from directory.")

        # Single file → add only if not already present
        elif p.is_file():
            if p.suffix == ".hdf5":
                on_load_calculations_clicked(b)
                refresh_selector()
                return
            if not (p.suffix in ['.h5', '.nxs', '.hdf']):
                return
            if already_in_df(p):
                print("File already in table:", p.name)
            else:
                new_row = file_metadata(p)
                DF_FILES.loc[len(DF_FILES)] = new_row
                print("Added file:", p.name)
        else:
            print("Path is neither file nor directory:", p)
            return

        refresh_selector()


def on_clean_clicked(b):
    global DF_FILES
    add_output.clear_output()   
    load_output.clear_output()

    DF_FILES = pandas.DataFrame(columns=["filename", "size_gb", "modified", "full_path"])
    on_clean_data_clicked(b)
    on_clean_calc_clicked(b)
    refresh_selector()


def on_clean_data_clicked(b):
    add_output.clear_output()   
    load_output.clear_output()
    DG_DATA.clear()


def on_clean_calc_clicked(b):
    add_output.clear_output()   
    load_output.clear_output()
    DG_CALC.clear()


def on_load_clicked(b):
    load_output.clear_output()
    with load_output:
        selected_indices = list(file_selector.value)
        if not selected_indices:
            print("No files selected.")
            return
        selected_rows = DF_FILES.loc[selected_indices]
        print("Files loading...")
        display(spinner)
        spinner.layout.display = "inline-block"

        for _, row in selected_rows.iterrows():
            file_path = row['full_path']
            if not pathlib.Path(file_path).is_file():
                continue
            try:
                load_dg_by_df_row(row)
                print(f" - {row['filename']} (row index: {row.name})")
                print(f"   full_path: {row['full_path']}")
            except Exception as e:
                print(f"Error reading file: {e}")
                continue
        spinner.layout.display = "none"


def on_reduce_clicked(b):
    load_output.clear_output()
    with load_output:
        selected_indices = list(file_selector.value)
        if not selected_indices:
            print("No files selected.")
            return
        selected_rows = DF_FILES.loc[selected_indices]
        print("Reducing Data...")
        display(spinner)
        spinner.layout.display = "inline-block"
        for _, row in selected_rows.iterrows():
            try:
                for s_key in D_DETECTOR_KEYS.keys():
                    _ = take_detector_reduced_calc_by_df_row(row, s_key, flag_calc=True)
            except Exception as e:
                print(f"Error reading file: {e}")
                continue
        spinner.layout.display = "none"


def on_tof_adjust_clicked(b):
    load_output.clear_output()
    with load_output:
        selected_indices = list(file_selector.value)
        if not selected_indices:
            print("No files selected.")
            return
        selected_rows = DF_FILES.loc[selected_indices]
        print("Adjusting TOF range based on Cave Monitor infomration...")
        display(spinner)
        spinner.layout.display = "inline-block"
        for _, row in selected_rows.iterrows():
            dg_magic = take_dg_by_df_row(row)
            da_cave_monitor = dg_magic.get('cave_monitor', None)
            if da_cave_monitor is None:
                continue
            flag = da_cave_monitor.data > da_cave_monitor.data * 0.1
            hh = da_cave_monitor.coords['toa'][flag]
            toa_min = hh.min().to(unit='ms')
            toa_max = hh.max().to(unit='ms')
            time_reduction_block['min'].value = numpy.round(toa_min.value,3)
            time_reduction_block['max'].value = numpy.round(toa_max.value,3)
            break
        print('Done')
        spinner.layout.display = "none"


def on_plot_laue_clicked(b):
    load_output.clear_output()
    with load_output:
        selected_indices = list(file_selector.value)
        if not selected_indices:
            print("No files selected.")
            return
        selected_rows = DF_FILES.loc[selected_indices]
        print("Plotting Laue Pattern...")
        display(spinner)
        spinner.layout.display = "inline-block"
        fig = None
        for _, row in selected_rows.iterrows():
            d_plot = {}
            vmax = 0.
            detector_a_laue_hist = take_detector_a_laue_hist_by_df_row(row)
            if not(detector_a_laue_hist is None):
                d_plot['detector_a'] = detector_a_laue_hist
                vmax = max([vmax, numpy.quantile(detector_a_laue_hist.data.values, 0.9)])
            detector_b_laue_hist = take_detector_b_laue_hist_by_df_row(row)
            if not(detector_b_laue_hist is None):
                d_plot['detector_b'] = detector_b_laue_hist
                vmax = max([vmax, numpy.quantile(detector_b_laue_hist.data.values, 0.9)])

            fig = plopp.scatter3d(
                d_plot,
                pos="event_position_global",
                cbar=True,
                size=0.005,
                opacity=0.75,
                vmax=vmax,
            )
            break
        spinner.layout.display = "none"
        if fig is None:
            return
        display_center(fig)


def on_plot_3d_graph_clicked(b):
    load_output.clear_output()
    fig = None
    da_peaks = None
    with load_output:
        (s_detector, s_normalization) = get_s_detector_and_normalization()
        s_ax_x, s_ax_y, s_ax_z = get_s_3d_ax()
        range_x, range_y, range_z = get_range_3d_ax()

        selected_indices = list(file_selector.value)
        if not selected_indices:
            print("No files selected.")
            return
        selected_rows = DF_FILES.loc[selected_indices]
        print("Plotting 3d Graph...")
        display(spinner)
        spinner.layout.display = "inline-block"
        for _, row in selected_rows.iterrows():
            hist_3d = get_hist_nd(row, s_detector, s_normalization, dims=(s_ax_x, s_ax_y, s_ax_z,), ranges=(range_x, range_y, range_z,))

            if hist_3d is None:
                continue
            if s_ax_x in D_DISPLAY_HIST_UNIT.keys():
                s_unit_display, s_unit_hist = D_DISPLAY_HIST_UNIT[s_ax_x]
                hist_3d = hist_3d.assign_coords(**{s_ax_x:hist_3d.coords[s_ax_x].to(unit=s_unit_display)})
            if s_ax_y in D_DISPLAY_HIST_UNIT.keys():
                s_unit_display, s_unit_hist = D_DISPLAY_HIST_UNIT[s_ax_y]
                hist_3d = hist_3d.assign_coords(**{s_ax_y:hist_3d.coords[s_ax_y].to(unit=s_unit_display)})
            if s_ax_z in D_DISPLAY_HIST_UNIT.keys():
                s_unit_display, s_unit_hist = D_DISPLAY_HIST_UNIT[s_ax_z]
                hist_3d = hist_3d.assign_coords(**{s_ax_z:hist_3d.coords[s_ax_z].to(unit=s_unit_display)})

            fig = plopp.inspector(
                hist_3d, 
                dim=s_ax_z,
                orientation="vertical",
                logc=False,
                mode="rectangle",
                autoscale=True,
            )

            if display_peaks_checkbox.value:
                threshold = threshold_boundfloattext.value
                binary_dilation = binary_dilation_boundinttext.value
                da_peaks = get_peaks(row, s_detector, s_normalization, dims=(s_ax_x, s_ax_y, s_ax_z,), ranges=(range_x, range_y, range_z,), threshold=threshold, binary_dilation=binary_dilation)
                if s_ax_x in D_DISPLAY_HIST_UNIT.keys():
                    s_unit_display, s_unit_hist = D_DISPLAY_HIST_UNIT[s_ax_x]
                    da_peaks = da_peaks.assign_coords(**{s_ax_x:da_peaks.coords[s_ax_x].to(unit=s_unit_display)})
                if s_ax_y in D_DISPLAY_HIST_UNIT.keys():
                    s_unit_display, s_unit_hist = D_DISPLAY_HIST_UNIT[s_ax_y]
                    da_peaks = da_peaks.assign_coords(**{s_ax_y:da_peaks.coords[s_ax_y].to(unit=s_unit_display)})
                if s_ax_z in D_DISPLAY_HIST_UNIT.keys():
                    s_unit_display, s_unit_hist = D_DISPLAY_HIST_UNIT[s_ax_z]
                    da_peaks = da_peaks.assign_coords(**{s_ax_z:da_peaks.coords[s_ax_z].to(unit=s_unit_display)})
            break

        spinner.layout.display = "none"
        if fig is None:
            return
        display_center(fig)
        if not(da_peaks is None):
            ax_1 = fig[0].fig.axes[0]
            ax_2 = fig[1].fig.axes[0]

            xlim_1 = ax_1.get_xlim()
            ylim_1 = ax_1.get_ylim()
            plopp.scatter(da_peaks, x=s_ax_x, y=s_ax_y, color='w', marker='.', alpha=0.5, ax=ax_1)
            ax_1.set_xlim(xlim_1)
            ax_1.set_ylim(ylim_1)

            xlim_2 = ax_2.get_xlim()
            ylim_2 = ax_2.get_ylim()
            ax_2.vlines(da_peaks.coords[s_ax_z].values, ylim_2[0], ylim_2[1], color="black", alpha=0.1)  
            ax_2.set_xlim(xlim_2)
            ax_2.set_ylim(ylim_2)


            display(scipp.table(da_peaks))


def on_plot_2d_graph_clicked(b):
    load_output.clear_output()
    fig = None
    with load_output:
        (s_detector, s_normalization) = get_s_detector_and_normalization()
        s_ax_x, s_ax_y = get_s_2d_ax()
        range_x, range_y = get_range_2d_ax()

        selected_indices = list(file_selector.value)
        if not selected_indices:
            print("No files selected.")
            return
        selected_rows = DF_FILES.loc[selected_indices]
        print("Plotting 2d Graph...")
        display(spinner)
        spinner.layout.display = "inline-block"
        for _, row in selected_rows.iterrows():
            hist_2d = get_hist_nd(row, s_detector, s_normalization, dims=(s_ax_x, s_ax_y, ), ranges=(range_x, range_y, ))

            if s_ax_x in D_DISPLAY_HIST_UNIT.keys():
                s_unit_display, s_unit_hist = D_DISPLAY_HIST_UNIT[s_ax_x]
                hist_2d = hist_2d.assign_coords(**{s_ax_x:hist_2d.coords[s_ax_x].to(unit=s_unit_display)})
            if s_ax_y in D_DISPLAY_HIST_UNIT.keys():
                s_unit_display, s_unit_hist = D_DISPLAY_HIST_UNIT[s_ax_y]
                hist_2d = hist_2d.assign_coords(**{s_ax_y:hist_2d.coords[s_ax_y].to(unit=s_unit_display)})

            if hist_2d is None:
                continue

            fig = plopp.plot(
                hist_2d, 
                coords=[s_ax_x, s_ax_y],
            )
            if display_peaks_checkbox.value:
                threshold = threshold_boundfloattext.value
                binary_dilation = binary_dilation_boundinttext.value
                # da_peaks = operations_with_da.find_peaks_hist(hist_2d, flag_variance=False, threshold=threshold, binary_dilation=binary_dilation)
                da_peaks = get_peaks(row, s_detector, s_normalization, dims=(s_ax_x, s_ax_y, ), ranges=(range_x, range_y, ), threshold=threshold, binary_dilation=binary_dilation)
                if s_ax_x in D_DISPLAY_HIST_UNIT.keys():
                    s_unit_display, s_unit_hist = D_DISPLAY_HIST_UNIT[s_ax_x]
                    da_peaks = da_peaks.assign_coords(**{s_ax_x:da_peaks.coords[s_ax_x].to(unit=s_unit_display)})
                if s_ax_y in D_DISPLAY_HIST_UNIT.keys():
                    s_unit_display, s_unit_hist = D_DISPLAY_HIST_UNIT[s_ax_y]
                    da_peaks = da_peaks.assign_coords(**{s_ax_y:da_peaks.coords[s_ax_y].to(unit=s_unit_display)})

                ax = fig.fig.axes[0]
                xlim = ax.get_xlim()
                ylim = ax.get_ylim()
                plopp.scatter(da_peaks, x=s_ax_x, y=s_ax_y, color='w', marker='.', alpha=0.5, ax=ax)
                ax.set_xlim(xlim)
                ax.set_ylim(ylim)
            break

        spinner.layout.display = "none"
        if fig is None:
            return
        display_center(fig)


def on_plot_1d_graph_clicked(b):
    load_output.clear_output()

    with load_output:
        (s_detector, s_normalization) = get_s_detector_and_normalization()
        s_ax_x = get_s_1d_ax()
        range_x = get_range_1d_ax()


        selected_indices = list(file_selector.value)
        if not selected_indices:
            print("No files selected.")
            return
        selected_rows = DF_FILES.loc[selected_indices]
        print("Plotting 1d Graph...")
        display(spinner)
        spinner.layout.display = "inline-block"
        d_plot = {}
        da_peaks = None
        for _, row in selected_rows.iterrows():
            hist_1d = get_hist_nd(row, s_detector, s_normalization, dims=(s_ax_x, ), ranges=(range_x, ))
            if hist_1d is None:
                continue
            if s_ax_x in D_DISPLAY_HIST_UNIT.keys():
                s_unit_display, s_unit_hist = D_DISPLAY_HIST_UNIT[s_ax_x]
                hist_1d = hist_1d.assign_coords(**{s_ax_x:hist_1d.coords[s_ax_x].to(unit=s_unit_display)})

            d_plot[str(row.name)] = hist_1d
            if display_peaks_checkbox.value:
                threshold = threshold_boundfloattext.value
                binary_dilation = binary_dilation_boundinttext.value
                # da_peaks = operations_with_da.find_peaks_hist(hist_1d, flag_variance=False, threshold=threshold, binary_dilation=binary_dilation)
                da_peaks = get_peaks(row, s_detector, s_normalization, dims=(s_ax_x, ), ranges=(range_x, ), threshold=threshold, binary_dilation=binary_dilation)
                if s_ax_x in D_DISPLAY_HIST_UNIT.keys():
                    s_unit_display, s_unit_hist = D_DISPLAY_HIST_UNIT[s_ax_x]
                    da_peaks = da_peaks.assign_coords(**{s_ax_x:da_peaks.coords[s_ax_x].to(unit=s_unit_display)})

        spinner.layout.display = "none"
        fig = plopp.plot(d_plot, coords=[s_ax_x, ])
        if not(da_peaks is None):
            ax = fig.fig.axes[0]
            xlim = ax.get_xlim()
            ylim = ax.get_ylim()
            ax.vlines(da_peaks.coords[s_ax_x].values, ylim[0], ylim[1], color="black", alpha=0.1)  
            ax.set_xlim(xlim)
            ax.set_ylim(ylim)
        display_center(fig)


def toggle_extra_reduction_checkbox(change):
    if change['new']:          # checkbox checked
        time_reduction_block['widget'].layout.display = 'block'
        count_reduction_block['widget'].layout.display = 'block'
        step_reduction_block['widget'].layout.display = "block"
        reduce_button.layout.display = 'block'
        tof_adjust_button.layout.display = "block"
    else:                      # checkbox unchecked
        time_reduction_block['widget'].layout.display = "none"
        count_reduction_block['widget'].layout.display = "none"
        step_reduction_block['widget'].layout.display = "none"
        reduce_button.layout.display = "none"
        tof_adjust_button.layout.display = "none"


def toggle_extra_plots_checkbox(change):
    if change['new']:          # checkbox checked
        detector_type_radio_button.layout.display = 'block'
        normalization_radio_button.layout.display = 'block'
        display_peaks_checkbox.layout.display = 'block'
        plot_graph_button.layout.display = 'block'
        plot_monitor_button.layout.display = 'block'
        axis_x_block['widget'].layout.display= "block"
        axis_y_block['widget'].layout.display= "block"
        axis_z_block['widget'].layout.display= "block"
        axis_spatial_block['widget'].layout.display= "block"
    else:                      # checkbox unchecked
        detector_type_radio_button.layout.display = 'none'
        normalization_radio_button.layout.display = 'none'
        display_peaks_checkbox.layout.display = 'none'
        plot_graph_button.layout.display = 'none'
        plot_monitor_button.layout.display = 'none'
        axis_x_block['widget'].layout.display= "none"
        axis_y_block['widget'].layout.display= "none"
        axis_z_block['widget'].layout.display= "none"
        axis_spatial_block['widget'].layout.display= "none"



def on_plot_graph_clicked(b):
    load_output.clear_output()
    with load_output:
        fig = scipp.show_graph({**magic_graphs.graph_detector, **magic_graphs.graph_qvec, **magic_graphs.graph_hkl}, simplified=True)
        display(fig)

        fig = scipp.show_graph(magic_graphs.graph_cave_monitor, simplified=True)
        display(fig)


def on_plot_monitor_clicked(b):
    load_output.clear_output()
    with load_output:
        selected_indices = list(file_selector.value)
        if not selected_indices:
            print("No files selected.")
            return
        selected_rows = DF_FILES.loc[selected_indices]
        print("Plotting Monitor information...")
        display(spinner)
        spinner.layout.display = "inline-block"

        s_ax = get_s_1d_ax()
        if not(s_ax in magic_graphs.graph_cave_monitor.keys()):
            s_ax = 'toa'

        d_plot = {}
        for _, row in selected_rows.iterrows():
            dg_magic = take_dg_by_df_row(row)
            da_cave_monitor = dg_magic.get('cave_monitor', None)
            if da_cave_monitor is None:
                continue
            da_cave_monitor.coords['cave_monitor_position'] = da_cave_monitor.coords['position']
            move_data_from_dg_to_da(dg_magic, da_cave_monitor)
            da_cave_monitor = da_cave_monitor.transform_coords((s_ax,), graph=magic_graphs.graph_cave_monitor)
            hist_1d = da_cave_monitor.hist(**{s_ax: 101})
            if s_ax in D_DISPLAY_HIST_UNIT.keys():
                s_unit_display, s_unit_hist = D_DISPLAY_HIST_UNIT[s_ax]
                hist_1d = hist_1d.assign_coords(**{s_ax:hist_1d.coords[s_ax].to(unit=s_unit_display)})

            d_plot[str(row.name)] = hist_1d
        spinner.layout.display = "none"
        fig = plopp.plot(
            d_plot, 
            coords=[s_ax, ],
        )            
        display_center(fig)


def on_find_peaks_clicked(b):
    load_output.clear_output()
    with load_output:
        selected_indices = list(file_selector.value)
        if not selected_indices:
            print("No files selected.")
            return
        selected_rows = DF_FILES.loc[selected_indices]
        print("Searching peaks...")
        display(spinner)
        spinner.layout.display = "inline-block"
        for _, row in selected_rows.iterrows():
            print(f" - File '{row.name:}'")
            for s_detector in ['da', 'db']:
                da_peaks_gamma_nu_toa = take_detector_peaks_gamma_nu_toa(row, s_detector)
                if da_peaks_gamma_nu_toa is None:
                    continue
                assign_peaks_to_reduced_data(row, da_peaks_gamma_nu_toa, s_detector)
                print(f"   Number of found peaks on detector {s_detector:} is '{da_peaks_gamma_nu_toa.coords["toa"].size:}'")
        spinner.layout.display = "none"



def toggle_extra_find_peaks_checkbox(change):
    if change['new']:          # checkbox checked
        threshold_boundfloattext.layout.display = 'block'
        binary_dilation_boundinttext.layout.display = 'block'
        load_peaks_button.layout.display = 'block'
        load_peaks_detector_type_radio_button.layout.display = 'block'
        fc_load_peaks.layout.display = 'block'
    else:                      # checkbox unchecked
        threshold_boundfloattext.layout.display = 'none'
        binary_dilation_boundinttext.layout.display = 'none'
        load_peaks_button.layout.display = 'none'
        load_peaks_detector_type_radio_button.layout.display = 'none'
        fc_load_peaks.layout.display = 'none'


def on_load_peaks_clicked(b):
    load_output.clear_output()
    with load_output:
        print("Loading peaks...")
        selected_indices = list(file_selector.value)
        if not selected_indices:
            print("No files selected for which peaks should be uploaded.")
            return
        selected_rows = DF_FILES.loc[selected_indices]
        if fc_load_peaks.value is None:
            print("Choose hdf5 file.")
            return

        p = pathlib.Path(fc_load_peaks.value).expanduser()

        if not p.exists():
            print("Path does not exist:", p)
            return
        if not p.is_file():
            print("Choose hdf5 file.")
            return
        if p.suffix != '.hdf5':
            print("Peaks should be save in .hdf5 format.")
            return
        da_peaks = scipp.io.load_hdf5(p.resolve())

        s_detector = "".join([hh[0] for hh in load_peaks_detector_type_radio_button.value.strip().split()]).lower()
        # FIXME: It should be expanded for arbitrary set of parameters and not only gamma, nu, toa (one of the options is qx, qy, qz) 
        s_key = f'peaks/{s_detector:}/nn/event_gamma/event_nu/toa'       
        for _, row in selected_rows.iterrows():
            dg_calc = take_dg_calc_by_df_row(row)
            dg_calc[s_key] = da_peaks
            assign_peaks_to_reduced_data(row, da_peaks, s_detector)


def on_save_calculations_clicked(b):
    global DG_CALC, DF_FILES
    load_output.clear_output()
    with load_output:
        print("Saving calculations...")
        display(spinner)
        spinner.layout.display = "inline-block"

        if fc_path_input.selected_path is None:
            filename = pathlib.Path(pathlib.Path('.').resolve(), 'magic_calc.hdf5') 
        else:
            filename = pathlib.Path(fc_path_input.selected_path, 'magic_calc.hdf5') 
        DG_CALC['File_Table'] = scipp.compat.pandas_compat.from_pandas(DF_FILES)
        DG_CALC.save_hdf5(filename=filename)
        spinner.layout.display = "none"
        print(f"The file has been saved to '{filename:}'.")


def on_load_calculations_clicked(b):
    global DG_CALC, DF_FILES
    load_output.clear_output()
    with load_output:
        print("Load calculations...")
        if fc_path_input.selected_path is None:
            filename = pathlib.Path(pathlib.Path('.').resolve(), 'magic_calc.hdf5') 
        else:
            filename = pathlib.Path(fc_path_input.selected_path, 'magic_calc.hdf5') 
        dg_calc = scipp.io.load_hdf5(filename)
        for key in dg_calc.keys():
            if key != 'File_Table':
                DG_CALC[key] = dg_calc[key]
            else:
                da_files =  dg_calc[key]
                for i in range(da_files.coords['row'].size):
                    d_raw = {key: item.value for key, item in da_files[i].items()} 
                    DF_FILES.loc[len(DF_FILES)] = d_raw
        print(f"The file has been loaded from '{filename:}'.")

# def update_ax_spatial_on_plot_hist_buttons(change):
#     if change['new'] == 'None':
#         plot_laue_button.description = "3D Laue"
#     else:
#         plot_laue_button.description = change['new']


def update_ax_x_on_plot_hist_buttons(change):
    if change['new'] == 'None':
        plot_hist_1d_button.description = "TOF"
        s_2d = "TOF"
        s_3d = "γ"
    else:
        plot_hist_1d_button.description = change['new']
        s_2d = change['new']
        s_3d = change['new']
    plot_hist_2d_button.description = "-".join([s_2d,]+plot_hist_2d_button.description.split('-')[1:])
    plot_hist_3d_button.description = "-".join([s_3d,]+plot_hist_3d_button.description.split('-')[1:])


def update_ax_y_on_plot_hist_buttons(change):
    if change['new'] == 'None':
        s_2d = "2θ"
        s_3d = "ν"
    else:
        s_2d = change['new']
        s_3d = change['new']
    plot_hist_2d_button.description = "-".join(plot_hist_2d_button.description.split('-')[:-1]+[s_2d,])
    l_hh = plot_hist_3d_button.description.split('-')
    l_hh[1] = s_3d
    plot_hist_3d_button.description = "-".join(l_hh)


def update_ax_z_on_plot_hist_buttons(change):
    if change['new'] == 'None':
        s_3d = "TOF"
    else:
        s_3d = change['new']
    plot_hist_3d_button.description = "-".join(plot_hist_3d_button.description.split('-')[:-1]+[s_3d,])


In [ ]:
# Signals

fc_path_input.register_callback(on_add_clicked)
load_button.on_click(on_load_clicked)
clean_button.on_click(on_clean_clicked)
clean_data_button.on_click(on_clean_data_clicked)
clean_calc_button.on_click(on_clean_calc_clicked)
extra_reduction_checkbox.observe(toggle_extra_reduction_checkbox, names='value')
reduce_button.on_click(on_reduce_clicked)
tof_adjust_button.on_click(on_tof_adjust_clicked)
filter.observe(on_filter_change, names='value')
plot_laue_button.on_click(on_plot_laue_clicked)
plot_hist_3d_button.on_click(on_plot_3d_graph_clicked)
plot_hist_2d_button.on_click(on_plot_2d_graph_clicked)
plot_hist_1d_button.on_click(on_plot_1d_graph_clicked)
extra_plots_checkbox.observe(toggle_extra_plots_checkbox, names='value')
plot_graph_button.on_click(on_plot_graph_clicked)
plot_monitor_button.on_click(on_plot_monitor_clicked)
axis_x_block['dropdown'].observe(update_ax_x_on_plot_hist_buttons, names="value")
axis_y_block['dropdown'].observe(update_ax_y_on_plot_hist_buttons, names="value")
axis_z_block['dropdown'].observe(update_ax_z_on_plot_hist_buttons, names="value")
# axis_spatial_block['dropdown'].observe(update_ax_spatial_on_plot_hist_buttons, names="value")
find_peaks_button.on_click(on_find_peaks_clicked)
extra_find_peaks_checkbox.observe(toggle_extra_find_peaks_checkbox, names='value')
load_peaks_button.on_click(on_load_peaks_clicked)
save_calculations_button.on_click(on_save_calculations_clicked)



In [ ]:
# Display

ipywidgets.VBox([
    fc_path_input, 
    add_output,
    ipywidgets.HBox([file_selector, 
                     ipywidgets.VBox([load_button, clean_button, clean_data_button, clean_calc_button, extra_reduction_checkbox]),]),
    filter,
    ipywidgets.HBox([time_reduction_block['widget'], ipywidgets.VBox([reduce_button, tof_adjust_button,]),]),
    count_reduction_block['widget'],
    step_reduction_block['widget'],
    ipywidgets.HTML("<b></b>"),
    ipywidgets.HBox([ipywidgets.HTML("<b>Graphs: </b>", layout=ipywidgets.Layout(width="150px")), plot_laue_button, plot_hist_3d_button, plot_hist_2d_button, plot_hist_1d_button, extra_plots_checkbox]),
    ipywidgets.HTML("<b></b>"),
    ipywidgets.HBox([detector_type_radio_button, normalization_radio_button, display_peaks_checkbox, plot_graph_button, plot_monitor_button]),
    axis_spatial_block["widget"],
    axis_x_block["widget"],
    axis_y_block["widget"],
    axis_z_block["widget"],
    ipywidgets.HTML("<b></b>"),
    ipywidgets.HTML("<b>Operations: </b>", layout=ipywidgets.Layout(width="150px")),
    ipywidgets.HBox([find_peaks_button, extra_find_peaks_checkbox]),
    ipywidgets.HBox([threshold_boundfloattext,binary_dilation_boundinttext,]),
    ipywidgets.HBox([fc_load_peaks, load_peaks_detector_type_radio_button, load_peaks_button]),
    ipywidgets.HBox([save_calculations_button,]),
    load_output,
])

In [ ]:
widget_code_cell.make_code_cell(
    width="80%", 
    placeholder="Write Python code here ...\n\nThe follwoing global variables are available:\n - DF_FILES \n - DG_DATA (stores loaded data) \n - DG_CALC (stores calculated results)",
    )